# JavaScript — Fetch & APIs

> **How this topic works**
> 1. **This notebook** — read the theory. Here the cells **really run**: `fetch`
>    works in this kernel, so you can hit a real API before touching the DOM.
> 2. **`project/`** — a real Vite app where you do the exercise in the browser.
>
> Read this first, then follow the steps at the bottom.

## LESSON 48 — Fetch & APIs

Everything from LESSON 42 (async/await) now pays off: the data lives on a server, and it arrives late.

```js
const response = await fetch("https://example.com/users");

if (!response.ok) {
  throw new Error(`Request failed with status ${response.status}`);
}

const users = await response.json();
```

### Three states, always

```
    loading   →   success (render the data)
              →   error   (tell the user what happened)
```

Beginners write only the success path. Then the app looks broken whenever the network is slow or down — which is most of the time on a phone.

### Where the code goes

Split it in two, and keep them apart:

- **`services/`** — knows URLs and JSON. Knows nothing about the page.
- the UI — knows the page. Contains **no URL**.

If the API ever moves, exactly one file changes. This is the `src/services/` folder you will build again in the final project, and the same split React projects use.

### Key notes

- **`fetch` does NOT throw on 404 or 500.** A server answering "not found" is, to `fetch`, a perfectly successful request. It only rejects when the network itself fails. **You** must check `response.ok`.
- **`response.json()` is async too** — it needs its own `await`.
- `await` only works inside an `async` function (or at a module's top level, which is why these cells work).
- Always wrap `await` in `try/catch`: a rejected promise throws.

### Keeping what you fetched

A page reload throws away every variable you had. `localStorage` is a small box of text the browser keeps for your site, and it survives reloads, tab closes and restarts.

```js
localStorage.setItem("theme", "dark");
localStorage.getItem("theme");        // "dark"
localStorage.getItem("missing");      // null
localStorage.removeItem("theme");
```

It stores **strings only**, which is where JSON from LESSON 38 comes back:

```js
localStorage.setItem("users", JSON.stringify(users));

const saved = JSON.parse(localStorage.getItem("users") ?? "[]");
```

That `?? "[]"` matters: `getItem` returns `null` for a key that was never set, and `JSON.parse(null)` gives you `null` rather than the empty array you were expecting.

`sessionStorage` has an identical API and forgets everything when the tab closes. Use it for state that should not outlive the visit.

A useful pattern: show the saved copy immediately, fetch in the background, then replace it. The page has something to display on the first frame instead of a spinner.

### Key notes

- **It stores strings only.** Store an object without `JSON.stringify` and you get back the text `"[object Object]"`.
- **`getItem` returns `null`, not `undefined`,** for a missing key. Give `JSON.parse` a fallback.
- Storage is per origin and roughly 5 MB. It is not a database, and never a place for anything secret — any script on the page can read it.
- Reading and writing are synchronous: they block the page. Fine for small values, bad in a loop.

### A real request — runnable

This calls [jsonplaceholder.typicode.com](https://jsonplaceholder.typicode.com):
public, no key, no rate limit. You need an internet connection.

In [ ]:
const response = await fetch("https://jsonplaceholder.typicode.com/users");

console.log("ok:", response.ok, "| status:", response.status);

const users = await response.json();

console.log("how many:", users.length);
console.log("first one:", users[0].name, "-", users[0].email);
console.log("city:", users[0].address.city);

### The trap, seen for real — runnable

A URL that doesn't exist. Predict what you'll see **before** running it: does the
`catch` fire?

In [ ]:
// Without the response.ok check
try {
  const bad = await fetch("https://jsonplaceholder.typicode.com/nope");
  console.log("reached the line after fetch — no error was thrown");
  console.log("but status is:", bad.status, "and ok is:", bad.ok);
} catch (error) {
  console.log("caught:", error.message);
}

console.log("---");

// With the check: now it behaves the way you expected in the first place
try {
  const bad = await fetch("https://jsonplaceholder.typicode.com/nope");

  if (!bad.ok) {
    throw new Error(`Request failed with status ${bad.status}`);
  }

  console.log("this line is never reached");
} catch (error) {
  console.log("caught:", error.message);
}

---

## Now do the exercise

**1. Start the project**

```bash
cd project
npm install     # only the first time
npm run dev
```

Open DevTools with **F12** and look at the **Network** tab this time, not just
the console.

**2. Read the live demo**

Two files, and the split between them is the lesson:
- `project/src/services/users-service.js` — the URLs live here
- `project/src/lessons/lesson-48-fetch.js` — the page logic, with no URL in sight

Click both demo buttons, including the broken one, and watch the status line.

**3. Do the exercise**

Open `project/src/exercise/exercise.js` and work through the numbered STEPs.
You'll create your own service file first.

**Done when:** clicking loads 10 post titles, you saw "Loading…" while it worked,
and deliberately breaking the URL shows an error message instead of a blank page.

Stuck? Paste `ai-prompt.txt` into a fresh AI session. The answer is in
`project/src/exercise/solution.js` — last resort.